**Como está composta a base de usuários da plataforma, quantos clientes, fornecedores e técnicos estão cadastrados, e como os técnicos se distribuem entre autônomos, filiados e parceiros?**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.requester AS
SELECT id, fk_company, business_type
FROM solaria.public.requester
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY business_type) = 1;

CREATE OR REPLACE TABLE workspace.silver.supplier AS
SELECT id, fk_company, status, business_type
FROM solaria.public.supplier
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY status) = 1;

CREATE OR REPLACE TABLE workspace.silver.technician_affiliation AS
SELECT id, fk_company, fk_technician, affiliation_type
FROM solaria.public.technician_affiliation
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY affiliation_type) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.fato_requesters AS
SELECT id AS requester_id, fk_company, business_type
FROM workspace.silver.requester;

CREATE OR REPLACE TABLE workspace.gold.fato_suppliers AS
SELECT id AS supplier_id, fk_company, status, business_type
FROM workspace.silver.supplier;

CREATE OR REPLACE TABLE workspace.gold.fato_tecnicos AS
SELECT
    t.id AS technician_id,
    ta.affiliation_type,
    ta.fk_company
FROM solaria.public.technician t
LEFT JOIN workspace.silver.technician_affiliation ta ON ta.fk_technician = t.id;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_composicao_atores AS
SELECT 'Cliente' AS tipo, COUNT(*) AS total FROM workspace.gold.fato_requesters
UNION ALL
SELECT 'Fornecedor', COUNT(*) FROM workspace.gold.fato_suppliers
UNION ALL
SELECT 'Técnico', COUNT(*) FROM workspace.gold.fato_tecnicos;

CREATE OR REPLACE TABLE workspace.gold.kpi_tecnicos_por_vinculo AS
SELECT
    COALESCE(affiliation_type, 'SEM_VINCULO') AS vinculo,
    COUNT(*) AS total,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS percentual
FROM workspace.gold.fato_tecnicos
GROUP BY COALESCE(affiliation_type, 'SEM_VINCULO');